# TP1 - Operaciones Morfologicas

**Materia:** Vision Artificial  
**Carrera:** Licenciatura en Automatizacion y Control  
**Facultad:** Universidad Tecnologica Nacional - Facultad Regional Cordoba  
**Trabajo Practico:** TP1 - Operaciones Morfologicas  
**Alumno:** Cristian Gonzalo Vera  
**Legajo:** 420581

Este notebook desarrolla de forma integral el TP1, aplicando una secuencia de procesamiento sobre una imagen real y una imagen sintetica. El desarrollo incluye conversion a escala de grises, binarizacion global mediante Otsu, analisis de histograma, operaciones morfologicas con kernel `3x3` y analisis de sensibilidad con kernel `5x1`.

## 1. Objetivo y metodologia de trabajo

El objetivo del TP es implementar y analizar una secuencia de operaciones morfologicas aplicadas al procesamiento de imagenes. Para ello se trabaja con dos casos complementarios:

- una **imagen real**, para observar el comportamiento del algoritmo en condiciones no ideales,
- una **imagen sintetica**, para validar el comportamiento teorico en un escenario controlado.

El pipeline aplicado a ambos casos es el siguiente:

1. conversion a escala de grises,
2. binarizacion global con Otsu,
3. analisis de histograma con umbral marcado,
4. erosion con kernel `3x3`,
5. dilatacion con kernel `3x3`,
6. apertura con kernel `3x3`,
7. cierre con kernel `3x3`,
8. repeticion del bloque morfologico con kernel `5x1`.

De este modo, el notebook no solo implementa el procesamiento, sino que documenta visualmente cada etapa y permite justificar los resultados obtenidos.

## 2. Preparacion del entorno

En esta seccion se importan las bibliotecas necesarias y se resuelven las rutas de entrada. El notebook esta preparado para trabajar con una carpeta `entrada/` ubicada junto al archivo `tp1.ipynb`.

In [ ]:
from pathlib import Path

import cv2
import numpy as np
from matplotlib import pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')


def resolve_input_dir() -> Path:
    candidates = [
        Path('entrada'),
        Path.cwd() / 'entrada',
        Path('/content/entrada'),
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError('No se encontro la carpeta entrada/.')


INPUT_DIR = resolve_input_dir()
REAL_IMAGE_PATH = INPUT_DIR / 'imagen_real.png'
SYNTHETIC_REFERENCE_PATH = INPUT_DIR / 'imagen_sintetica_referencia.png'

print('Carpeta de entrada:', INPUT_DIR)
print('Imagen real:', REAL_IMAGE_PATH)
print('Imagen sintetica de referencia:', SYNTHETIC_REFERENCE_PATH)

## 3. Funciones auxiliares

Para mantener el notebook legible, se encapsulan las tareas repetitivas en funciones. Estas funciones permiten:

- cargar y visualizar imagenes,
- generar la imagen sintetica por codigo,
- aplicar el pipeline completo de procesamiento,
- y resumir metricas simples de area blanca para comparar resultados.

In [ ]:
KERNEL_3X3 = np.ones((3, 3), dtype=np.uint8)
KERNEL_5X1 = np.ones((1, 5), dtype=np.uint8)


def load_color_image(path: Path) -> np.ndarray:
    image = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if image is None:
        raise FileNotFoundError(f'No se pudo leer la imagen: {path}')
    return image


def bgr_to_rgb(image_bgr: np.ndarray) -> np.ndarray:
    return cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)


def show_color(ax, image_bgr: np.ndarray, title: str) -> None:
    ax.imshow(bgr_to_rgb(image_bgr))
    ax.set_title(title)
    ax.axis('off')


def show_gray(ax, image_gray: np.ndarray, title: str) -> None:
    ax.imshow(image_gray, cmap='gray', vmin=0, vmax=255)
    ax.set_title(title)
    ax.axis('off')


def plot_histogram(ax, gray: np.ndarray, threshold: float, title: str) -> None:
    hist = cv2.calcHist([gray], [0], None, [256], [0, 256]).flatten()
    ax.plot(hist, color='black', linewidth=1.5)
    ax.axvline(threshold, color='red', linestyle='--', linewidth=2, label=f'Otsu = {threshold:.2f}')
    ax.set_title(title)
    ax.set_xlabel('Nivel de gris')
    ax.set_ylabel('Cantidad de pixeles')
    ax.legend()


def create_synthetic_image(width: int = 900, height: int = 600) -> np.ndarray:
    image = np.zeros((height, width, 3), dtype=np.uint8)

    cv2.rectangle(image, (80, 140), (320, 420), (0, 220, 220), thickness=-1)
    cv2.rectangle(image, (150, 210), (250, 320), (0, 0, 0), thickness=-1)

    cv2.circle(image, (560, 250), 95, (255, 220, 0), thickness=-1)

    cv2.rectangle(image, (460, 390), (540, 450), (255, 120, 0), thickness=-1)
    cv2.rectangle(image, (565, 390), (645, 450), (255, 120, 0), thickness=-1)

    for center in ((700, 120), (735, 145), (770, 170), (190, 500), (230, 525)):
        cv2.circle(image, center, 8, (255, 255, 255), thickness=-1)

    return image


def apply_pipeline(image_bgr: np.ndarray) -> dict:
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    threshold, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    morph_3x3 = {
        'erosion': cv2.erode(binary, KERNEL_3X3, iterations=1),
        'dilatacion': cv2.dilate(binary, KERNEL_3X3, iterations=1),
        'apertura': cv2.morphologyEx(binary, cv2.MORPH_OPEN, KERNEL_3X3),
        'cierre': cv2.morphologyEx(binary, cv2.MORPH_CLOSE, KERNEL_3X3),
    }

    morph_5x1 = {
        'erosion': cv2.erode(binary, KERNEL_5X1, iterations=1),
        'dilatacion': cv2.dilate(binary, KERNEL_5X1, iterations=1),
        'apertura': cv2.morphologyEx(binary, cv2.MORPH_OPEN, KERNEL_5X1),
        'cierre': cv2.morphologyEx(binary, cv2.MORPH_CLOSE, KERNEL_5X1),
    }

    metrics = {
        'Binaria Otsu': int(np.count_nonzero(binary)),
        'Erosion 3x3': int(np.count_nonzero(morph_3x3['erosion'])),
        'Dilatacion 3x3': int(np.count_nonzero(morph_3x3['dilatacion'])),
        'Apertura 3x3': int(np.count_nonzero(morph_3x3['apertura'])),
        'Cierre 3x3': int(np.count_nonzero(morph_3x3['cierre'])),
        'Erosion 5x1': int(np.count_nonzero(morph_5x1['erosion'])),
        'Dilatacion 5x1': int(np.count_nonzero(morph_5x1['dilatacion'])),
        'Apertura 5x1': int(np.count_nonzero(morph_5x1['apertura'])),
        'Cierre 5x1': int(np.count_nonzero(morph_5x1['cierre'])),
    }

    return {
        'original': image_bgr,
        'gray': gray,
        'threshold': threshold,
        'binary': binary,
        'morph_3x3': morph_3x3,
        'morph_5x1': morph_5x1,
        'metrics': metrics,
    }


def print_metrics(title: str, metrics: dict) -> None:
    print(title)
    print('-' * len(title))
    for key, value in metrics.items():
        print(f'{key:<18}: {value}')


## 4. Caso de estudio: imagen real

La imagen real se utiliza para estudiar el comportamiento del algoritmo en condiciones no ideales. En este tipo de imagenes es habitual encontrar pequenas variaciones de iluminacion, bordes menos definidos y detalles que no aparecen en una escena sintetica.

Por esa razon, este caso permite evaluar la robustez practica del pipeline exigido por el TP.

In [ ]:
real_bgr = load_color_image(REAL_IMAGE_PATH)
real_results = apply_pipeline(real_bgr)

fig, ax = plt.subplots(1, 1, figsize=(6, 6))
show_color(ax, real_bgr, 'Imagen real original')
plt.show()

### 4.1 Conversion a escala de grises, Otsu e histograma

La conversion a escala de grises reduce la imagen a un unico canal de intensidad. Este paso es critico porque la binarizacion global mediante Otsu se aplica sobre niveles de gris, no directamente sobre color.

En terminos tecnicos, OpenCV utiliza una ponderacion de canales para aproximar la luminosidad percibida. Luego, el metodo de Otsu analiza el histograma de la imagen y calcula automaticamente el umbral que mejor separa dos clases.

En el histograma se marca el valor de umbral obtenido, lo que permite evaluar si la separacion entre fondo y objeto fue razonable.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
show_color(axes[0, 0], real_results['original'], 'Original')
show_gray(axes[0, 1], real_results['gray'], 'Escala de grises')
show_gray(axes[1, 0], real_results['binary'], 'Mascara binaria con Otsu')
plot_histogram(axes[1, 1], real_results['gray'], real_results['threshold'], 'Histograma con umbral Otsu')
plt.tight_layout()
plt.show()

print(f"Umbral calculado por Otsu para la imagen real: {real_results['threshold']:.2f}")

### 4.2 Operaciones morfologicas con kernel `3x3`

Con el kernel cuadrado `3x3` se espera un comportamiento relativamente uniforme en todas las direcciones. Las operaciones se interpretan de la siguiente manera:

- **Erosion:** retrae bordes y ayuda a eliminar pequenas regiones blancas aisladas.
- **Dilatacion:** expande regiones blancas y tiende a cerrar pequenas brechas.
- **Apertura:** combina erosion seguida de dilatacion, por lo que es util para remover ruido pequeno.
- **Cierre:** combina dilatacion seguida de erosion, por lo que es util para reforzar continuidad y rellenar huecos pequenos.

En una imagen real, estas transformaciones permiten observar como el objeto segmentado cambia de forma en presencia de irregularidades propias de la escena.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
show_gray(axes[0], real_results['morph_3x3']['erosion'], 'Erosion 3x3')
show_gray(axes[1], real_results['morph_3x3']['dilatacion'], 'Dilatacion 3x3')
show_gray(axes[2], real_results['morph_3x3']['apertura'], 'Apertura 3x3')
show_gray(axes[3], real_results['morph_3x3']['cierre'], 'Cierre 3x3')
plt.tight_layout()
plt.show()

### 4.3 Analisis de sensibilidad con kernel `5x1`

Para completar el TP se repiten las operaciones morfologicas relevantes con un kernel rectangular `5x1`. A diferencia del `3x3`, este elemento estructurante introduce una preferencia direccional horizontal.

Esa diferencia permite analizar anisotropia: el resultado deja de ser completamente uniforme en todas las direcciones y pasa a depender mas fuertemente de la orientacion de huecos, bordes y uniones.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
show_gray(axes[0], real_results['morph_5x1']['erosion'], 'Erosion 5x1')
show_gray(axes[1], real_results['morph_5x1']['dilatacion'], 'Dilatacion 5x1')
show_gray(axes[2], real_results['morph_5x1']['apertura'], 'Apertura 5x1')
show_gray(axes[3], real_results['morph_5x1']['cierre'], 'Cierre 5x1')
plt.tight_layout()
plt.show()

print_metrics('Metricas del caso real', real_results['metrics'])

### 4.4 Conclusion del caso real

La imagen real confirma que el pipeline funciona en un escenario no ideal. La binarizacion con Otsu logra una mascara util y las operaciones morfologicas muestran el comportamiento esperado:

- la erosion reduce la superficie blanca y retrae el contorno,
- la dilatacion aumenta la superficie blanca y refuerza conectividad,
- la apertura elimina detalles pequenos,
- y el cierre tiende a reforzar continuidad rellenando pequenas discontinuidades.

La diferencia entre `3x3` y `5x1` es visible pero moderada, lo cual es coherente con una imagen real donde el efecto del kernel se mezcla con ruido, textura e iluminacion. Justamente por eso este caso es importante: permite evaluar el algoritmo en condiciones mas cercanas a una aplicacion practica.

## 5. Caso de estudio: imagen sintetica

La imagen sintetica permite controlar el escenario y validar el comportamiento teorico del algoritmo. En este notebook se muestra primero la referencia disponible en `entrada/`, y luego se genera una escena propia por codigo con fondo negro, figuras geometricas y pequenos detalles pensados para que las operaciones morfologicas produzcan efectos interpretablemente claros.

In [ ]:
synthetic_reference_bgr = load_color_image(SYNTHETIC_REFERENCE_PATH)
synthetic_generated_bgr = create_synthetic_image()
synthetic_results = apply_pipeline(synthetic_generated_bgr)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
show_color(axes[0], synthetic_reference_bgr, 'Referencia sintetica')
show_color(axes[1], synthetic_generated_bgr, 'Imagen sintetica generada por codigo')
plt.tight_layout()
plt.show()

### 5.1 Conversion a escala de grises, Otsu e histograma

En el caso sintetico, la conversion a grises y la binarizacion suelen ofrecer una respuesta mas estable que en la imagen real, ya que el fondo es uniforme y el contraste entre figuras y fondo es alto. Esto permite verificar el comportamiento teorico del metodo de Otsu en condiciones favorables.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
show_color(axes[0, 0], synthetic_results['original'], 'Sintetica generada')
show_gray(axes[0, 1], synthetic_results['gray'], 'Escala de grises')
show_gray(axes[1, 0], synthetic_results['binary'], 'Mascara binaria con Otsu')
plot_histogram(axes[1, 1], synthetic_results['gray'], synthetic_results['threshold'], 'Histograma con umbral Otsu')
plt.tight_layout()
plt.show()

print(f"Umbral calculado por Otsu para la imagen sintetica: {synthetic_results['threshold']:.2f}")

### 5.2 Operaciones morfologicas con kernel `3x3`

En la imagen sintetica, el kernel cuadrado `3x3` actua sobre figuras bien definidas y con poco ruido incidental. Esto hace que la interpretacion sea mas directa:

- la erosion reduce claramente el espesor de las figuras,
- la dilatacion las expande,
- la apertura remueve pequenos puntos blancos agregados como ruido controlado,
- y el cierre preserva o refuerza la continuidad de las estructuras principales.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
show_gray(axes[0], synthetic_results['morph_3x3']['erosion'], 'Erosion 3x3')
show_gray(axes[1], synthetic_results['morph_3x3']['dilatacion'], 'Dilatacion 3x3')
show_gray(axes[2], synthetic_results['morph_3x3']['apertura'], 'Apertura 3x3')
show_gray(axes[3], synthetic_results['morph_3x3']['cierre'], 'Cierre 3x3')
plt.tight_layout()
plt.show()

### 5.3 Analisis de sensibilidad con kernel `5x1`

El kernel `5x1` permite observar mejor el comportamiento direccional. En esta escena se incluyo deliberadamente una pequena brecha horizontal entre componentes, de modo que la anisotropia se vuelva interpretable.

Al repetir erosion, dilatacion, apertura y cierre con este kernel, se obtiene una comparacion directa con el resultado mas uniforme del `3x3`.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
show_gray(axes[0], synthetic_results['morph_5x1']['erosion'], 'Erosion 5x1')
show_gray(axes[1], synthetic_results['morph_5x1']['dilatacion'], 'Dilatacion 5x1')
show_gray(axes[2], synthetic_results['morph_5x1']['apertura'], 'Apertura 5x1')
show_gray(axes[3], synthetic_results['morph_5x1']['cierre'], 'Cierre 5x1')
plt.tight_layout()
plt.show()

print_metrics('Metricas del caso sintetico', synthetic_results['metrics'])

### 5.4 Conclusion del caso sintetico

La imagen sintetica valida el comportamiento teorico del pipeline en condiciones controladas. La segmentacion es limpia, el histograma responde de forma mas clara y las operaciones morfologicas muestran el efecto esperado sobre figuras geometricas y detalles pequenos.

En este contexto, la diferencia entre `3x3` y `5x1` se interpreta mejor porque hay menos ruido incidental. Por eso el caso sintetico complementa al real: permite entender la teoria con mayor claridad y luego contrastarla con un escenario mas cercano a la practica.

## 6. Comparacion entre kernels y entre casos

Para respaldar el analisis visual, se comparan las cantidades de pixeles blancos obtenidas en la binaria inicial y en las salidas morfologicas. Esta metrica simple no reemplaza la observacion cualitativa, pero ayuda a confirmar tendencias:

- la erosion debe reducir superficie blanca,
- la dilatacion debe incrementarla,
- la apertura suele remover pequenos detalles,
- y el cierre tiende a consolidar regiones y rellenar huecos pequenos.

Ademas, la comparacion entre `3x3` y `5x1` permite discutir anisotropia.

In [ ]:
real_labels = list(real_results['metrics'].keys())
real_values = list(real_results['metrics'].values())
synthetic_labels = list(synthetic_results['metrics'].keys())
synthetic_values = list(synthetic_results['metrics'].values())

fig, axes = plt.subplots(2, 1, figsize=(14, 10))
axes[0].bar(real_labels, real_values, color='steelblue')
axes[0].set_title('Cantidad de pixeles blancos - Caso real')
axes[0].tick_params(axis='x', rotation=45)

axes[1].bar(synthetic_labels, synthetic_values, color='darkorange')
axes[1].set_title('Cantidad de pixeles blancos - Caso sintetico')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print('Resumen comparativo')
print('-------------------')
print(f"Caso real - Binaria Otsu: {real_results['metrics']['Binaria Otsu']}")
print(f"Caso real - Erosion 3x3 / 5x1: {real_results['metrics']['Erosion 3x3']} / {real_results['metrics']['Erosion 5x1']}")
print(f"Caso real - Dilatacion 3x3 / 5x1: {real_results['metrics']['Dilatacion 3x3']} / {real_results['metrics']['Dilatacion 5x1']}")
print()
print(f"Caso sintetico - Binaria Otsu: {synthetic_results['metrics']['Binaria Otsu']}")
print(f"Caso sintetico - Erosion 3x3 / 5x1: {synthetic_results['metrics']['Erosion 3x3']} / {synthetic_results['metrics']['Erosion 5x1']}")
print(f"Caso sintetico - Dilatacion 3x3 / 5x1: {synthetic_results['metrics']['Dilatacion 3x3']} / {synthetic_results['metrics']['Dilatacion 5x1']}")

## 7. Conclusiones finales

A partir del desarrollo realizado, se pueden sintetizar las siguientes conclusiones:

1. La conversion a escala de grises es un paso indispensable porque simplifica la informacion visual y permite aplicar correctamente la binarizacion.
2. El metodo de Otsu resulta adecuado para segmentar ambas imagenes, aunque su efectividad depende mas fuertemente del histograma en el caso real.
3. La erosion, la dilatacion, la apertura y el cierre exhiben el comportamiento teorico esperado, modificando la forma y conectividad de los objetos binarios.
4. El kernel `3x3` produce una accion mas uniforme, mientras que el kernel `5x1` introduce una respuesta direccional asociada a anisotropia.
5. La imagen real permite estudiar el algoritmo en una situacion mas cercana a una aplicacion practica, mientras que la imagen sintetica facilita validar la teoria en condiciones controladas.
6. La comparacion entre ambos casos justifica el enfoque experimental del TP: relacionar teoria y practica a partir de evidencia visual y analitica.

En conjunto, el trabajo muestra que las operaciones morfologicas no solo modifican la superficie blanca de una mascara binaria, sino tambien la estructura geometrica de los objetos en funcion del elemento estructurante utilizado.